## 1. Import libraries and load dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [2]:
train_df  = pd.read_csv("/content/twitter_training.csv", header=None)
val_df  = pd.read_csv("/content/twitter_validation.csv", header=None)

## 2. Adding a header

In [3]:
train_df.columns = ['id', 'entity', 'sentiment', 'text']
val_df.columns = ['id', 'entity', 'sentiment', 'text']

## 3. Dataset info

In [4]:
train_df.shape

(74682, 4)

In [5]:
train_df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [6]:
train_df.sample(5)

,id,entity,sentiment,text
64407,7834,MaddenNFL,Neutral,@EAMaddenNFL not what you will care.
14654,2911,Dota2,Negative,why is it always drow?: (rip hero with less set
60292,3531,Facebook,Neutral,"Because I always keep reading, endless reading..."
779,2537,Borderlands,Neutral,Even tho i successfully leveled this grenade i...
65620,6837,johnson&johnson,Negative,THE SA avoid this by try any means please don'...


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         74682 non-null  int64 
 1   entity     74682 non-null  object
 2   sentiment  74682 non-null  object
 3   text       73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [8]:
train_df.isnull().sum()

,0
id,0
entity,0
sentiment,0
text,686


In [9]:
train_df['sentiment'].value_counts()

,count
sentiment,
Negative,22542
Positive,20832
Neutral,18318
Irrelevant,12990


## 4. Drop missing text (train + validation)

In [10]:
train_df = train_df.dropna(subset=['text']).reset_index(drop=True)
val_df = val_df.dropna(subset=['text']).reset_index(drop=True)

## 4. Define features and labels

In [11]:
X_train = train_df['text']
y_train = train_df['sentiment']

X_val = val_df['text']
y_val = val_df['sentiment']


## 6. Naive Bayes pipeline

In [12]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2)
    )),
    ("nb", MultinomialNB(alpha=1.0))
])


## 7. Train the model

In [13]:
nb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('nb', MultinomialNB())])

## 8. Validate the model

In [14]:
y_pred = nb_pipeline.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_val, y_pred))

Validation Accuracy: 0.894

Classification Report:

              precision    recall  f1-score   support

  Irrelevant       0.99      0.82      0.90       172
    Negative       0.83      0.95      0.89       266
     Neutral       0.94      0.84      0.89       285
    Positive       0.87      0.94      0.90       277

    accuracy                           0.89      1000
   macro avg       0.91      0.89      0.89      1000
weighted avg       0.90      0.89      0.89      1000



## 8. Predict on new samples

In [15]:
samples = [
    "I love this so much",
    "This is absolutely terrible",
    "It is fine, nothing special",
    "Click here to win a prize",
    "It is so ugly"
]

nb_pipeline.predict(samples)

array(['Positive', 'Negative', 'Positive', 'Neutral', 'Negative'],
      dtype='<U10')